[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fw-ai/cookbook/blob/main/training/case-studies/grpo_countdown/countdown_grpo.ipynb)


# Serverless non-agentic GRPO on Countdown — the loop you own

Reinforcement learning with **GRPO** on a **single-turn**, automatically graded
task — done on the Fireworks **serverless** Training API. You write (and see)
the loop; Fireworks does not run managed RFT for you.

This notebook **imports** the existing loop in
`training/examples/serverless_rl/countdown_rl.py`. It does not paste GRPO math.

## Pipeline

1. Configure a smoke-sized run and validate the dataset schema.
2. Sanity-check the reward on fixed strings (**no GPU**).
3. Show the GRPO primitives on a fake reward group (still free).
4. Optionally run a short live loop + held-out eval (`RUN_LIVE = True`).
5. Show the **same algorithm** expressed for dedicated GPUs (config swap — the
   same serverless/dedicated side-by-side as [`sft_prompt_router`](../sft_prompt_router/)).

## Prerequisites

```bash
# from the cookbook `training/` directory (preferred — pulls tinker + the cookbook package)
uv sync
# or: pip install -e ".[eval]" && pip install python-dotenv datasets matplotlib
```

`.env` (repo `training/.env` or cookbook root): `FIREWORKS_API_KEY`, `FIREWORKS_ACCOUNT_ID`.

> Live train/eval cells spend serverless tokens. Leave `RUN_LIVE = False` until
> you intend to pay. Serverless teardown is a no-op.


## 1. Setup & credentials

Imports, `.env`, and path setup so `training.examples.serverless_rl` resolves
whether the kernel started in this folder or at the repo root.


In [ ]:
import json
import os
import sys
from pathlib import Path

import dotenv

# Locate this case study, then put the cookbook root on sys.path.
# (.env discovery comes after: find_dotenv walks *parents* of the cwd, so a
# kernel started at the repo root would never see training/.env — we probe
# the documented locations explicitly.)
_HERE = next(
    (
        p
        for p in [
            Path.cwd(),
            Path.cwd() / "training/case-studies/grpo_countdown",
            Path.cwd() / "case-studies/grpo_countdown",
        ]
        if (p / "notebook_helpers.py").exists()
    ),
    None,
)
assert _HERE is not None, (
    "launch from training/case-studies/grpo_countdown/ (or the cookbook repo root)"
)
_REPO_ROOT = next(
    (
        p
        for p in [_HERE, *_HERE.parents]
        if (p / "training" / "examples" / "serverless_rl" / "countdown_rl.py").exists()
    ),
    None,
)
assert _REPO_ROOT is not None, "could not find the cookbook repo root"

dotenv.load_dotenv(dotenv.find_dotenv(usecwd=True), override=True)
# First match wins: training/.env is the documented primary location; loading
# both would let a repo-root .env override the training credentials.
for _env_path in (_REPO_ROOT / "training" / ".env", _REPO_ROOT / ".env"):
    if _env_path.exists():
        dotenv.load_dotenv(_env_path, override=True)
        break

for _p in (str(_HERE), str(_REPO_ROOT)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from notebook_helpers import (  # noqa: E402
    composite_reward,
    format_reward,
    group_relative_advantages,
    load_countdown_rl,
    prepare_dataset,
    smoke_config,
)

assert os.getenv("FIREWORKS_API_KEY"), "FIREWORKS_API_KEY missing from .env"
ACCOUNT_ID = os.getenv("FIREWORKS_ACCOUNT_ID", "").replace("accounts/", "", 1)
assert ACCOUNT_ID, "FIREWORKS_ACCOUNT_ID missing from .env"

print(f"SDK env ready. account: {ACCOUNT_ID}")
print(f"case study: {_HERE}")
print(f"repo root:  {_REPO_ROOT}")


## 2. Which path are you on?

Fireworks has three customization shapes that matter for RL. This notebook is
the **serverless, non-agentic** cell of the grid.

| | Non-agentic (single-turn, graded) | Agentic (tools / multi-turn) |
| --- | --- | --- |
| **Managed** | [`reasoning_rl`](../reasoning_rl/) — one RFT job, no loop | — |
| **Serverless** | **this notebook** (Countdown) | [`agentic_rl_text2sql`](../agentic_rl_text2sql/) |
| **Dedicated** | §10 below → [`recipes/rl_loop.py`](../../recipes/rl_loop.py) | Harbor [`examples/rl/harbor/`](../../examples/rl/harbor/) |

Serverless → dedicated is a **path swap** (same GRPO). Non-agentic → agentic is a
**different product** (harness). Do not nest both axes into four notebooks.

On serverless there is nothing to provision: one
`FiretitanServiceClient(.../training/v1/serverless)` yields both the LoRA
training client and per-snapshot samplers.


## 3. Configuration

Smoke defaults stay small on purpose. Flip `RUN_LIVE` only when you want a real
serverless session. Raise `STEPS` / `GROUP_SIZE` when you want a visible climb.


In [ ]:
# --- gates -----------------------------------------------------------------
RUN_LIVE = False              # True -> create a serverless session and train
PREPARE_FULL_DATASET = False  # True -> download TinyZero rows (HF) once

# --- model (must be in the serverless trainer pool) ------------------------
BASE_MODEL = "accounts/fireworks/models/qwen3-8b"
TOKENIZER_MODEL = "Qwen/Qwen3-8B"
RENDERER_NAME = ""  # empty = auto-resolve from tokenizer

# For Kimi K3 instead (MoE + Router Replay), use:
# BASE_MODEL = "accounts/fireworks/models/kimi-k3"
# TOKENIZER_MODEL = "moonshotai/Kimi-K3"
# os.environ["HF_TRUST_REMOTE_CODE"] = "1"

# --- paths -----------------------------------------------------------------
SAMPLE_DATASET = _HERE / "countdown_train.sample.jsonl"
FULL_DATASET = _HERE / "countdown_3to4_train.jsonl"
RUN_DIR = _HERE / "runs" / "smoke"

# Materialize a larger file only when asked (gitignored). Needs only `datasets`.
if PREPARE_FULL_DATASET:
    prepare_dataset(FULL_DATASET, num_rows=256, seed=0)

DATASET = FULL_DATASET if FULL_DATASET.exists() else SAMPLE_DATASET
assert DATASET.exists(), f"missing dataset: {DATASET}"

n_rows = sum(1 for line in DATASET.open() if line.strip())
print(
    f"RUN_LIVE={RUN_LIVE} dataset={DATASET.name} rows≈{n_rows}\n"
    f"smoke defaults (applied when cfg is built): steps=2 groups=2 group_size=4 eval_holdout=4"
)

# Build countdown_rl.Config only when we need the runner (imports tinker).
# Gated on RUN_LIVE alone: PREPARE_FULL_DATASET needs only `datasets`, not tinker.
cfg = None
if RUN_LIVE:
    cfg = smoke_config(
        dataset=DATASET,
        run_dir=RUN_DIR,
        base_model=BASE_MODEL,
        tokenizer_model=TOKENIZER_MODEL,
        renderer_name=RENDERER_NAME,
    )
    print(f"cfg ready: steps={cfg.steps} group_size={cfg.group_size}")
else:
    print("cfg deferred — free cells below do not need it. Flip RUN_LIVE to build.")


## 4. Data validation (before you spend)

Every row needs chat `messages` plus a parseable `ground_truth` with `numbers`
and `target`. Catch schema mistakes here — not after you open a session.


In [ ]:
rows = [json.loads(line) for line in DATASET.open() if line.strip()]
assert rows, f"empty dataset: {DATASET}"

required_msg_keys = {"role", "content"}
for i, row in enumerate(rows):
    assert "messages" in row and "ground_truth" in row, f"row {i}: need messages + ground_truth"
    assert isinstance(row["messages"], list) and row["messages"], f"row {i}: empty messages"
    for m in row["messages"]:
        assert required_msg_keys <= set(m), f"row {i}: message missing role/content"
    # The live loop's eval does json.loads(row["ground_truth"]) and then reads
    # gt["numbers"] / gt["target"] — so require exactly that form here. A dict
    # (or a "nums" row) would pass a looser check and then crash the live eval.
    gt_raw = row["ground_truth"]
    assert isinstance(gt_raw, str), (
        f"row {i}: ground_truth must be a JSON-encoded string "
        '(the loop json.loads it), e.g. \'{"numbers": [...], "target": N}\''
    )
    gt = json.loads(gt_raw)
    # isinstance first: "in" on a str does substring matching, so a JSON string
    # like '"numbers target"' would otherwise sneak past the key checks.
    assert isinstance(gt, dict), f"row {i}: ground_truth must decode to an object"
    assert "numbers" in gt and "target" in gt, f"row {i}: bad ground_truth (need numbers/target)"

print(f"validated {len(rows)} rows")
print("example user:", rows[0]["messages"][-1]["content"][:160], "...")
print("ground_truth:", rows[0]["ground_truth"])


## 5. Reward sanity — no GPU

`composite_reward` is what the loop optimizes: partial credit for a parseable
`<answer>`, using exactly the source numbers, and hitting the target. If this
cell looks wrong, fix the reward before paying for rollouts.


In [ ]:
gt = {"numbers": [1, 2, 3, 4], "target": 24}

cases = [
    ("perfect", "<think>product</think>\n<answer>1 * 2 * 3 * 4</answer>"),
    ("right numbers, wrong value", "<think>x</think>\n<answer>1 + 2 + 3 + 4</answer>"),
    ("wrong numbers", "<think>x</think>\n<answer>8 * 3</answer>"),
    ("no answer tag", "<think>I give up</think>"),
]

print(f"{'case':32} {'format':>7} {'composite':>10}")
for label, text in cases:
    print(
        f"{label:32} {format_reward(text):7.2f} {composite_reward(text, gt):10.2f}"
    )


## 6. GRPO primitives (teaching — still free)

Within one prompt's group, standardize rewards to `(r - mean) / std`. Better-
than-average completions get positive advantage (pushed up); worse get negative.
**Zero-spread** groups (every sample scores the same) yield no signal — the
real loop drops them.

The production implementation is imported from `countdown_rl.py` as
`group_relative_advantages` (re-exported by `notebook_helpers`). We do not
re-derive the loss here. This cell imports `countdown_rl` (needs `uv sync` / `tinker`).

Each live step then does:

1. `save_weights_for_sampler` — snapshot the current LoRA for sampling
2. sample `group_size` completions per prompt
3. score → within-group advantages → drop zero-spread
4. `forward_backward(..., "importance_sampling")` + `optim_step`


In [ ]:
# Fake group — same math as countdown_rl (imported), no API calls.
# Requires the cookbook training install (`uv sync`) so `tinker` is available.
fake_group = [
    ("hits target with right numbers", 1.0),
    ("format + numbers, wrong value", 0.3),
    ("parseable answer only", 0.1),
    ("empty / garbage", 0.0),
]
labels = [x for x, _ in fake_group]
rewards = [r for _, r in fake_group]
advs = group_relative_advantages(rewards)

mean = sum(rewards) / len(rewards)
print(f"group mean = {mean:.3f}\n")
print(f"{'completion':34} {'reward':>7} {'advantage':>10}")
for lbl, r, a in zip(labels, rewards, advs):
    arrow = "push UP  " if a > 0 else ("push DOWN" if a < 0 else "no change")
    print(f"{lbl:34} {r:>7.2f} {a:>10.3f}   {arrow}")

flat = [0.3, 0.3, 0.3, 0.3]
print(f"\nzero-spread {flat} -> {group_relative_advantages(flat)}")
print("=> all ~0 advantages: the live loop skips this group (no gradient).")


## 7. Short live loop + W&B (optional)

When `RUN_LIVE = True`, this constructs `ServerlessCountdownRL(cfg)` and calls
`.run()`. That class owns:

- serverless session + LoRA training client
- fixed holdout carve-out (`eval_prompt_groups`) scored at start / interval / end
- metrics JSONL (+ W&B if you set `cfg.wandb_entity` / `WANDB_API_KEY`)
- optional final checkpoint promotion via `cfg.output_model_id`

Leave the gate off while you are still learning the cells above.


In [ ]:
records = []
runner = None

if not RUN_LIVE:
    print("RUN_LIVE=False — skipping serverless session. Flip the gate in §3 to train.")
else:
    if cfg is None:
        cfg = smoke_config(
            dataset=DATASET,
            run_dir=RUN_DIR,
            base_model=BASE_MODEL,
            tokenizer_model=TOKENIZER_MODEL,
            renderer_name=RENDERER_NAME,
        )
    # Optional W&B: set these before constructing the runner.
    # from dataclasses import replace
    # cfg = replace(cfg, wandb_entity="your-entity", wandb_project="countdown-grpo")
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    ServerlessCountdownRL = load_countdown_rl().ServerlessCountdownRL
    runner = ServerlessCountdownRL(cfg)
    print(
        f"session={runner.session_name or runner.session_id} run={runner.run_id}\n"
        f"train_rows={len(runner.rows)} eval_rows={len(runner.eval_rows)}"
    )
    records = runner.run()
    print(f"finished {len(records)} optimizer steps; metrics in {runner.run_dir}")


## 8. After-eval & optional promote

The runner already evals the **fixed holdout** during `.run()` when
`eval_at_start` / `eval_interval` / `eval_at_end` are set (they are, in
`smoke_config`). Inspect `eval_metrics.jsonl` under `run_dir`.

To promote the final sampler checkpoint to a Fireworks LoRA model, set
`output_model_id` on `Config` before `ServerlessCountdownRL` is constructed
(see `countdown_rl.py`). Promotion is skipped when that field is empty.


In [ ]:
if runner is None:
    print("No live run — nothing to summarize.")
else:
    eval_path = runner.eval_metrics_path
    if eval_path.exists():
        eval_rows = [json.loads(line) for line in eval_path.open() if line.strip()]
        for rec in eval_rows:
            print(
                f"eval after_steps={rec.get('completed_steps')} "
                f"reward={rec.get('eval/raw_reward')}"
            )
    else:
        print(f"no eval metrics at {eval_path}")
    print(f"run_dir={runner.run_dir}")


## 9. Cleanup

**Serverless:** there is no trainer job and no inference deployment to tear
down. Closing the process is enough; leftover sampler snapshots are session-
scoped.

**Dedicated:** see §10 — you must clean up the trainer job and hot-load
deployment (or set `cleanup_on_exit=True` on `rl_loop.Config`).


In [ ]:
# Serverless path: nothing to delete.
print("serverless cleanup: no-op (no trainer job / deployment on this path)")

# If you promoted an output model or created other account-scoped resources
# outside this notebook, delete them explicitly with the Fireworks SDK / firectl.


## 10. The same GRPO on dedicated (config swap — does not execute)

Nothing below runs. It is the **same algorithm** (group rollouts → advantages →
policy update) expressed on the dedicated Training API, the same way
[`sft_prompt_router`](../sft_prompt_router/) puts the serverless and dedicated
paths side by side for SFT.

On dedicated you provision a **trainer job** and an **inference deployment**,
then hot-load weight snapshots between them. The cookbook recipe is
`training/recipes/rl_loop.py` (sync) or `async_rl_loop.py` (overlapped).

```python
from dataclasses import replace
from training.recipes.rl_loop import Config as DedicatedGRPOConfig
from training.utils import DeployConfig, TrainerConfig

# Same task shape: JSONL rows + a local reward. One fork is required for
# Countdown: rl_loop's built-in reward_fn expects the target as <answer>N</answer>
# in the row itself, but Countdown rows carry {"numbers": [...], "target": N} —
# as-is every reward is 0.0 and every group is dropped. In your fork of
# recipes/rl_loop.py, replace reward_fn with the Countdown reward:
#
#   from training.examples.serverless_rl.countdown_rewards import composite_reward
#
#   def reward_fn(completion: str, row: dict) -> float:
#       return float(composite_reward(completion, row["ground_truth"]))
dedicated_cfg = DedicatedGRPOConfig(
    log_path="/tmp/countdown-dedicated.jsonl",
    base_model="accounts/fireworks/models/qwen3-8b",
    dataset=str(DATASET),          # same Countdown JSONL
    completions_per_prompt=4,      # ≈ serverless group_size
    prompt_groups_per_step=2,      # ≈ serverless prompt_groups_per_step
    max_completion_tokens=512,
    learning_rate=1e-4,
    lora_rank=32,
    epochs=1,
    max_rows=32,
    trainer=TrainerConfig(
        # List shapes with: firectl training-shape list --no-paginate
        training_shape_id="accounts/fireworks/trainingShapes/<your-shape>",
    ),
    deployment=DeployConfig(tokenizer_model="Qwen/Qwen3-8B"),
    cleanup_on_exit=True,          # tear down trainer + deployment when the recipe exits
)
```

| | Serverless (this notebook) | Dedicated (`rl_loop`) |
| --- | --- | --- |
| Wait before step 1 | seconds (attach to pool) | ~10 min provisioning |
| Billing | per token | GPU-hours while up |
| You manage | session only | trainer job + deployment + hot-load |
| Teardown | no-op | `cleanup_on_exit=True` or delete both |

**Agentic dedicated** does not collapse into this section — use Harbor
(`training/examples/rl/harbor/`, e.g. `train_pi.py`) when you need a sandbox
harness and multi-turn token traces.


## Recap

- **Owned the loop** on serverless: snapshot → sample group → score → GRPO
  advantages → `forward_backward` + `optim_step`, imported from `countdown_rl.py`.
- **Validated** data and reward before spending.
- **Dedicated** is the same GRPO with different compute (`rl_loop`), not a
  second notebook.
- **Managed RFT** and **agentic** paths already have their own case studies —
  do not fold them in here.

### Related

- Managed GRPO RFT: `case-studies/reasoning_rl/`
- Serverless agentic GRPO: `case-studies/agentic_rl_text2sql/`
- Script form of this loop: `examples/serverless_rl/countdown_rl.py`
- Dedicated sync/async GRPO: `recipes/rl_loop.py`, `recipes/async_rl_loop.py`
- Harbor (agentic at scale): `examples/rl/harbor/`
